In [1]:
import os
#
import tableauserverclient as TSC

In [ ]:

tableau_server_url = "https://10ay.online.tableau.com/"
site_id = "lcomreporting"
personal_access_token_name = os.environ.get("Script_Token_Name")
personal_access_token_secret = os.environ.get("Script_Token_Secret")
datasource_name = "Fivetran User Not Found"
target_datasource_id = None
extract_refresh_task_id = None

In [3]:
# --- 1. Authenticate to Tableau Server ---
tableau_auth = TSC.PersonalAccessTokenAuth(
    token_name=personal_access_token_name,
    personal_access_token=personal_access_token_secret,
    site_id=site_id
)
server = TSC.Server(tableau_server_url, use_server_version=True)

In [4]:
# --- 2.Data Source ID by Name
try:
    with server.auth.sign_in(tableau_auth):
        all_datasources, pagination_item = server.datasources.get()        
        for datasource in all_datasources:
            if datasource.name == datasource_name:
                target_datasource_id = datasource.id
                break

        if target_datasource_id:
            print("Data source ID is:", target_datasource_id)
        else:
            print("Data source not found.")

except TSC.ServerResponseError as e:
    print(f"Tableau Server Error: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")            

Data source not found.


In [5]:
# --- 3. Associated Extract Task by Data Source ID
if target_datasource_id:
    try:
        with server.auth.sign_in(tableau_auth):
            
            all_tasks, pagination_item = server.tasks.get()
            associated_tasks = []
            for task in all_tasks:
                if task.target and task.target.id == target_datasource_id:
                    associated_tasks.append(task)

            if associated_tasks:
                for task in associated_tasks:
                    extract_refresh_task_id = task.id
                    print(f"Task ID: {task.id}, Task Type: {task.task_type}")
            else:
                print("No tasks found associated with the data source.")

    except TSC.ServerResponseError as e:
        print(f"Tableau Server Error: {e}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")             

In [6]:
# --- 4. Run the Extract Refresh Task ---
if extract_refresh_task_id:
    try:
        with server.auth.sign_in(tableau_auth):
        
            print(f"Attempting to run extract refresh task with ID: {extract_refresh_task_id}")

            extract_refresh_task = server.tasks.get_by_id(extract_refresh_task_id)  # Get the task object
            server.tasks.run(extract_refresh_task)  # Run the task

            print(f"Successfully triggered extract refresh task {extract_refresh_task_id}")
            print("Note: The refresh task is added to the backgrounder queue and might not run immediately.")
            print("You can query the job status using the Query Job method and the job ID returned by this operation.") # This is a general point, specific job ID isn't directly returned by this run method in the simplified example.

    except TSC.ServerResponseError as e:
        print(f"Tableau Server Error: {e}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")